In [3]:
import weaviate
from weaviate.classes.config import Configure, Multi2VecField

In [4]:
client = weaviate.connect_to_local()

In [6]:
# Multi2VecField?

Init signature: Multi2VecField(*, name: str, weight: Optional[float] = None) -> None
Docstring:      Use this class when defining the fields to use in the `Multi2VecClip` and `Multi2VecBind` vectorizers.
Init docstring:
Create a new model by parsing and validating input data from keyword arguments.

Raises [`ValidationError`][pydantic_core.ValidationError] if the input data cannot be
validated to form a valid model.

`self` is explicitly positional-only to allow `self` as a field name.
File:           c:\users\swast\appdata\local\programs\python\python312\lib\site-packages\weaviate\collections\classes\config_vectorizers.py
Type:           ModelMetaclass
Subclasses:     

In [5]:
# Configure.Vectorizer.multi2vec_clip?

Signature:
Configure.Vectorizer.multi2vec_clip(
    image_fields: Union[List[str], List[weaviate.collections.classes.config_vectorizers.Multi2VecField], NoneType] = None,
    text_fields: Union[List[str], List[weaviate.collections.classes.config_vectorizers.Multi2VecField], NoneType] = None,
    interference_url: Optional[str] = None,
    inference_url: Optional[str] = None,
    vectorize_collection_name: bool = True,
) -> weaviate.collections.classes.config_vectorizers._VectorizerConfigCreate
Docstring:
Create a `_Multi2VecClipConfigCreate` object for use when vectorizing using the `multi2vec-clip` model.

See the [documentation](https://weaviate.io/developers/weaviate/modules/retriever-vectorizer-modules/multi2vec-clip)
for detailed usage.

Arguments:
    `image_fields`
        The image fields to use in vectorization.
    `text_fields`
        The text fields to use in vectorization.
    `inference_url`
        The inference url to use where API requests should go. Defaults to `None

In [7]:
if client.collections.exists('ClipCollection'):
        # collection = client.collections.get('ClipCollection')
        client.collections.delete('ClipCollection')
# else:
collection = client.collections.create(
    name="ClipCollection",
    vectorizer_config=Configure.Vectorizer.multi2vec_clip(
            image_fields=[
                Multi2VecField(
                        name="image"
                )
            ],
            text_fields=[
                Multi2VecField(
                        name="text"
                )
            ]
    ),
    generative_config=Configure.Generative.ollama(
        api_endpoint="http://host.docker.internal:11434",
        model="llama3.1"
    )
)

In [8]:
from data_loader import load_data

object_list = load_data()


In [9]:
object_list[0]

{'chunk_no': 1,
 'text': 'Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, “and what is the use of a book,” thought Alice “without pictures or conversations?”',
 'file_path': 'c:\\Users\\swast\\OneDrive\\Desktop\\files_working\\topics.txt',
 'media_type': 'text'}

In [10]:
# with collection.batch.dynamic() as batch:
#     for object in object_list:
#         batch.add_object(
#             properties=object
#         )
collection.data.insert(object_list[0])

UUID('3f223460-e983-472a-9e22-1b001ed932f6')

In [11]:
collection.query.fetch_objects()

QueryReturn(objects=[Object(uuid=_WeaviateUUIDInt('3f223460-e983-472a-9e22-1b001ed932f6'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'text': 'Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, “and what is the use of a book,” thought Alice “without pictures or conversations?”', 'media_type': 'text', 'file_path': 'c:\\Users\\swast\\OneDrive\\Desktop\\files_working\\topics.txt', 'chunk_no': 1.0}, references=None, vector={}, collection='ClipCollection')])

In [ ]:

class Database:
    
    def __init__(self):
        self.create_client()
        self.generate_collection()
        # self.data_ingestion()
        # self.initiate_interaction()
        # self.close_connection()

    def create_client(self):
        self.client = weaviate.connect_to_local()

    def generate_collection(self):
        if self.client.collections.exists('ClipCollection'):
            self.collection = self.client.collections.get('ClipCollection')
            return
        self.collection = self.client.collections.create(
            name="ClipCollection",
            vectorizer_config=Configure.Vectorizer.multi(
                api_endpoint="http://host.docker.internal:11434",
                model="nomic-embed-text"
            ),
            generative_config=Configure.Generative.ollama(
                api_endpoint="http://host.docker.internal:11434",
                model="llama3.1"
            )
        )
    
    def data_ingestion(self):
        object_list = data_loader.load_data()
        with self.collection.batch.dynamic() as batch:
            for object in object_list:
                batch.add_object(
                    properties=object
                )
    
    def search_with_text(self, query : str):
        return self.collection.query.hybrid(
            query=query,
            limit=3
        )

    def generate_with_text(self, query : str):
        return self.collection.generate.near_text(
            query=query,
            limit=3,
            grouped_task="You are an helpful AI Assistant who explains the given text",
            grouped_properties=['text']
        )

    def initiate_interaction(self):
        while True:
            print("-" * 20)
            choice = int(input("0 Exit, 1 Search, 2 Generate : "))
            if choice == 0:
                print("Exiting")
                break
            elif choice == 1:
                query = input("Query : ")
                response = self.search_with_text(query)
                for object in response.objects:
                    print(object.properties)
            elif choice == 2:
                query = input("Query : ")
                response = self.generate_with_text(query)
                print(response.generated)
                print("--SOURCE--")
                for object in response.objects:
                    print(object.properties)

    def close_connection(self):
        self.client.close()